# 🚀 Getting Started with Kaggle Benchmarks

Welcome! This notebook will teach you how to create, run, and evaluate LLM benchmarks using the `kaggle-benchmarks` library.

**Key concepts** 
1. Task: A Python function defining the problem (e.g., "Solve this riddle").
2. Run: The execution of a task
3. Benchmark: A collection of tasks that is arbitrarily put together by a user. There is no code implementation for this. This is a feature that Kaggle supports on the graphical user interface so that users can put together their own benchmarks based on the tasks that they care about

Now, let's dive into creating a task and executing your first run!


In [ ]:
# We import the library as 'kbench' for brevity
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

print("Ready to benchmark!")

# Part 1: Creating Your First Task

Here, we define the task (`@kbench.task`). All logic lives inside a single Python function and it acts as a container for the:

- 🗣️ Prompt (llm.prompt): The input. You ask the model a question or give it a command. (e.g., What gets wtter as it dries?)
- ⚖️ Verify: The check. How you determine if the LLM's answer was correct. An easy way to do this is with an assertion (e.g., assert that "Towel" is in the response)
- 📝 Return (return ...): The score. You return a value to determine the final grade on the leaderboard. If no value is returned, the task is graded Pass/Fail based on its assertions.

In [ ]:
@kbench.task(name="solve_riddle")
def solve_riddle(llm, riddle: str, answer: str) -> dict:
    # 1. Prompt the LLM
    response = llm.prompt(riddle)
    print(f"Model Answer: {response}")

    # 2. Grade the response (simple string check instead of Regex)
    is_correct = answer.lower() in response.lower()

    # 3. Assert based on the boolean calculation
    kbench.assertions.assert_true(
        is_correct,
        expectation=f"The model's answer should contain '{answer}'."
    )

    # 4. Set a return value (optional, but useful for batch evaluation - see part 2)
    return {
        "is_correct": is_correct,
        "model_response": response
    }

# Run the task immediately to test it
# kbench.llm is the default model pre-loaded in this environment
solve_riddle.run(
    llm=kbench.llm,
    riddle="What gets wetter as it dries?",
    answer="Towel",
)

# Part 2: Scaling Up (Batch Evaluation)

Running one question is useful for testing, but benchmarks usually involve evaluating a model across a Dataset.

**The `.evaluate()` method**
Instead of running `.run()` once, we can use `.evaluate()` to run our task over a pandas DataFrame.

Important: To score a dataset, your task needs to return a value.
- Return a bool (True/False) for simple accuracy.
- Return int or float for a specific score (0-100).

Below, we modify our task to return a bool so we can calculate an accuracy percentage.

In [ ]:
# 1. Create a small dataset
df = pd.DataFrame([
    {"riddle": "What has keys but can't open locks?", "answer": "Piano"},
    {"riddle": "What has an eye but cannot see?", "answer": "Needle"},
    {"riddle": "I shave every day, but my beard stays the same. What am I?", "answer": "Barber"}
])

# 2. Define a scoring task (returns an accuracy score)
@kbench.task(name="batch_riddle_solver")
def score_riddle_accuracy(llm, df) -> float:
    # Enable caching to speed up development and avoid re-running identical queries
    with kbench.client.enable_cache():
        # Execute the 'solve_riddle' task for every row in our dataframe
        runs = solve_riddle.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],  # Ensure the evaluation runs until all rows in the dataframe are processed
            max_attempts=1, # Limit retries to 1 to fail fast during testing
            llm=[llm], # Pass the specific LLM we want to evaluate
            evaluation_data=df,
            n_jobs=3, # Run 3 examples in parallel to significantly speed up the benchmark
        )

    # Convert the raw run objects into a pandas DataFrame for easy analysis
    eval_df = runs.as_dataframe()

    # Calculate the average success rate by taking the mean of the 'is_correct' column
    accuracy = float(eval_df.result.str.get("is_correct").mean())
    # Return the final calculated accuracy
    return accuracy

In [ ]:
_ = score_riddle_accuracy.run(kbench.llm, df)

Congratulations! You've now run your first task over a dataset.  

# Part 3: Choose the Task for your Task Detail page

Kaggle Benchmarks requires you to specify one primary task to populate your Task Detail Page, which is created when you hit "Save Task" on the top right hand corner of this notebook.

Run the cell below to lock in `batch_riddle_solver` (instead of `solve_riddle`) as your submitted task. You can change this later by pointing %choose to a different task function.

In [ ]:
%choose batch_riddle_solver

# (Optional) Part 4: Advanced Features
Now that you have the basics, here are some powerful features to create more types of tasks.
- A. Complex Inputs (Vision, Multi-turn)
- B. Advanced Logic (Agents/Tools, Multi-Model Comparison)
- C. Deep Evaluation (Return Types, LLM-as-a-Judge)